# Melatih model deteksi SMS penipuan (smishing) → TensorFlow Lite

Notebook ini melatih model teks ringan dari `sample_dataset.csv`, lalu mengekspornya ke
**TFLite** agar bisa dijalankan di aplikasi Flutter *tanpa internet*.

Jalankan di **Google Colab** (gratis, bisa pakai GPU) atau lokal dengan `pip install tensorflow pandas scikit-learn`.

Dataset contoh ini kecil (hanya untuk demo). Untuk akurasi nyata, perbanyak datanya
(kumpulkan contoh SMS penipuan berbahasa Indonesia, idealnya ratusan–ribuan baris).

In [ ]:
import pandas as pd, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
print('TensorFlow', tf.__version__)

In [ ]:
# 1. Muat data
df = pd.read_csv('sample_dataset.csv')
print(df['label'].value_counts())
X_train, X_val, y_train, y_val = train_test_split(
    df['text'].astype(str), df['label'].astype('float32'),
    test_size=0.2, random_state=42, stratify=df['label'])
len(X_train), len(X_val)

In [ ]:
# 2. Vektorisasi teks (jadi bagian dari model, supaya aplikasi cukup kirim teks mentah)
VOCAB = 2000
MAXLEN = 40
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB, output_mode='int', output_sequence_length=MAXLEN)
vectorizer.adapt(X_train.values)

In [ ]:
# 3. Model kecil (embedding + global average + dense)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(VOCAB, 16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 4. Latih
history = model.fit(
    X_train.values, y_train.values,
    validation_data=(X_val.values, y_val.values),
    epochs=40, batch_size=8, verbose=2)

In [ ]:
# 5. Uji cepat
tests = [
    'Selamat anda pemenang undian, klaim dengan PIN ATM anda',
    'Bu besok kumpul tugas jam 9 di lab ya',
    'Mohon sebutkan kode OTP anda ke petugas',
]
for t in tests:
    p = float(model.predict([t], verbose=0)[0][0])
    print(f'{p:.2f}  {t}')

In [ ]:
# 6. Ekspor ke TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite = converter.convert()
open('smishing.tflite','wb').write(tflite)
print('Tersimpan: smishing.tflite', len(tflite), 'bytes')

## Mengintegrasikan ke aplikasi Flutter (opsional, peningkatan)

Aplikasi `awas_scam` sudah berfungsi penuh dengan mesin berbasis pola di
`lib/services/detector.dart`. Model TFLite ini adalah **lapisan tambahan** agar deteksi lebih pintar.

Langkah integrasi:
1. Salin `smishing.tflite` ke `assets/` dan daftarkan di `pubspec.yaml` (`flutter: assets:`).
2. Tambah paket `tflite_flutter` (dan `TextVectorization` sudah menyatu di model, jadi input cukup string).
3. Buat `MlDetector` yang memuat model dan mengembalikan probabilitas 0..1.
4. Gabungkan skornya dengan skor pola, mis. `skor_akhir = 0.6*pola + 0.4*ml`.

Simpan model dan dataset yang kamu perbanyak, lalu latih ulang untuk hasil lebih baik.